# 01. 산업 시나리오 기반 문제 정의 — PCB AOI 결함 검출

## 학습 목표
1. **산업 현장 시나리오**를 이해하고 Vision AI 과제로 번역한다.
2. **비즈니스 문제 → ML 문제** 변환 프레임워크를 적용한다.
3. 성공 지표, 제약 조건, 데이터 요구사항을 정의한다.

---

## 1. 산업 시나리오: 스마트 팩토리 PCB 검사 라인

### 현장 배경
| 항목 | 내용 |
|------|------|
| **공정** | PCB 제조 후 **AOI(Automated Optical Inspection)** 공정 |
| **설비** | 컨베이어 + 고해상도 산업용 카메라 + 링 조명 |
| **현재 방식** | 숙련 검사원 육안 검사 → 피로도, 검출률 편차, 병목 |
| **목표** | **Vision AI**로 6종 미세 결함을 실시간 자동 검출 |

### 6종 결함 유형
| ID | 클래스 | 산업적 의미 | 품질 영향 |
|----|--------|------------|----------|
| 0 | `mouse_bite` | 배선 가장자리 파손 | 신호 무결성 저하 |
| 1 | `spur` | 불필요 돌기 | 단락 위험 |
| 2 | `missing_hole` | 드릴 누락 | 부품 실장 불가 |
| 3 | `short` | 배선 단락 | 회로 오동작 |
| 4 | `open_circuit` | 배선 단선 | 기능 상실 |
| 5 | `spurious_copper` | 잔류 구리 | 단락·절연 파괴 |

## 2. 비즈니스 문제 → ML 문제 변환

### 2-1. 문제 정의 템플릿

아래 표를 채워 **과제를 구체화**합니다.

| 구분 | 질문 | 본 프로젝트 답안 |
|------|------|----------------|
| **Who** | 누가 이 문제를 겪는가? | AOI 공정 품질팀, 생산 라인 운영자 |
| **What** | 무엇이 문제인가? | 6종 미세 결함의 누락·오검출 |
| **When** | 언제 발생하는가? | 매 보드 통과 시(라인 속도 30~60 boards/min) |
| **Where** | 어디서 해결하는가? | AOI 검사 스테이션 엣지 GPU |
| **Why** | 왜 AI인가? | 인력 한계, 24h 일관성, 데이터 축적 가능 |
| **How** | 어떻게 측정하는가? | mAP, Recall(불량 미검), FPR(양품 오판) |

In [1]:
from pathlib import Path
import yaml

# 프로젝트 루트 (notebooks/ 기준 상위)
PROJECT_ROOT = Path("..").resolve()
CONFIG_PATH = PROJECT_ROOT / "config" / "dataset.yaml"

with open(CONFIG_PATH, encoding="utf-8") as f:
    dataset_cfg = yaml.safe_load(f)

print("=== 프로젝트 정의서 ===")
print(f"프로젝트명 : Smart Factory PCB AOI 결함 검출")
print(f"데이터 경로: {PROJECT_ROOT / dataset_cfg['path']}")
print(f"결함 클래스: {dataset_cfg['nc']}종")
for idx, name in dataset_cfg["names"].items():
    desc = dataset_cfg["class_descriptions"].get(name, "")
    print(f"  [{idx}] {name:16s} — {desc}")

=== 프로젝트 정의서 ===
프로젝트명 : Smart Factory PCB AOI 결함 검출
데이터 경로: C:\MyCursorLab\10_프로젝트\data\pcb-defect-dataset\pcb-defect-dataset
결함 클래스: 6종
  [0] mouse_bite       — 배선 가장자리가 깨져 들어간 결함
  [1] spur             — 배선에서 불필요하게 돌출된 가는 돌기
  [2] missing_hole     — 드릴링이 누락되거나 불완전한 구멍
  [3] short            — 인접 배선 간 의도치 않은 단락
  [4] open_circuit     — 배선이 끊어져 전기적 개방 상태
  [5] spurious_copper  — 설계에 없는 불필요한 구리 잔류물


## 3. ML 문제 유형 결정

### 후보 과제 유형 비교

| 접근법 | 입력 | 출력 | 장점 | 단점 | 본 프로젝트 |
|--------|------|------|------|------|------------|
| **이진 분류** | 보드 이미지 | OK/NG | 단순, 빠름 | 결함 위치·유형 불명 | ✗ |
| **다중 분류** | 결함 패치 | 6-class | 유형 구분 | 위치 정보 없음 | ✗ |
| **객체 검출** | 보드 이미지 | bbox + class | 위치·유형 동시 | 라벨링 비용 | **✓ 채택** |
| **세그멘테이션** | 보드 이미지 | 픽셀 마스크 | 정밀 윤곽 | 라벨 비용 매우 큼 | ✗ |

**최종 ML 문제 정의**
> **입력**: AOI 카메라가 촬영한 PCB RGB 이미지 (256×256 또는 600×600)
> **출력**: 결함 바운딩 박스 좌표 + 6-class 라벨 (YOLO 형식)
> **학습 방식**: 지도학습 (Supervised Object Detection)

In [2]:
# ML 문제 정의서를 구조화된 dict로 작성
ml_problem_definition = {
    "task_type": "object_detection",
    "input_modality": "vision_rgb",
    "input_shape": "H×W×3 (256 or 600)",
    "output_format": "YOLO bbox (class_id, cx, cy, w, h) normalized",
    "num_classes": 6,
    "annotation_format": "YOLO txt per image",
    "deployment_target": "edge_gpu_on_aoi_line",
    "latency_requirement_ms": 50,
    "success_metrics": {
        "primary": "mAP@0.5",
        "business_critical": "Recall ≥ 99% (불량 미검 최소화)",
        "secondary": "Precision (과검 최소화)",
    },
    "constraints": [
        "조명 변화(light_01~12)에 강건해야 함",
        "회전 증강(rotation_*) 환경 대응",
        "실시간 처리 (30+ FPS 목표)",
        "라벨 품질이 모델 성능의 상한을 결정",
    ],
}

import json
print(json.dumps(ml_problem_definition, indent=2, ensure_ascii=False))

{
  "task_type": "object_detection",
  "input_modality": "vision_rgb",
  "input_shape": "H×W×3 (256 or 600)",
  "output_format": "YOLO bbox (class_id, cx, cy, w, h) normalized",
  "num_classes": 6,
  "annotation_format": "YOLO txt per image",
  "deployment_target": "edge_gpu_on_aoi_line",
  "latency_requirement_ms": 50,
  "success_metrics": {
    "primary": "mAP@0.5",
    "business_critical": "Recall ≥ 99% (불량 미검 최소화)",
    "secondary": "Precision (과검 최소화)"
  },
  "constraints": [
    "조명 변화(light_01~12)에 강건해야 함",
    "회전 증강(rotation_*) 환경 대응",
    "실시간 처리 (30+ FPS 목표)",
    "라벨 품질이 모델 성능의 상한을 결정"
  ]
}


## 4. KPI 및 ROI 산정 (실습)

### 4-1. 비즈니스 KPI 정의

| KPI | 정의 | 목표값 | 측정 방법 |
|-----|------|--------|----------|
| **검출 Recall** | 실제 불량 중 AI가 잡은 비율 | ≥ 99% | Confusion Matrix |
| **오검 FPR** | 양품을 불량으로 판정 | ≤ 2% | FP / (FP + TN) |
| **처리 속도** | 보드 1장 검사 시간 | ≤ 33ms | 벤치마크 |
| **리워크 비용 절감** | 불량 유출 방지 | 월 X% | 불량 유출 건수 추적 |

### 4-2. 📝 실습: ROI 간이 계산

아래 변수를 수정하여 **연간 절감액**을 계산해 보세요.

In [3]:
# === ROI 계산 실습 (값을 바꿔가며 시나리오 분석) ===
boards_per_day = 10_000          # 일일 검사 보드 수
defect_rate = 0.03               # 불량률 3%
human_miss_rate = 0.05           # 육안 검사 미검률 5%
ai_miss_rate = 0.01              # AI 도입 후 미검률 1%
cost_per_escape = 50_000         # 불량 유출 1건당 비용 (원)
working_days = 250

human_escapes = boards_per_day * defect_rate * human_miss_rate * working_days
ai_escapes = boards_per_day * defect_rate * ai_miss_rate * working_days

savings = (human_escapes - ai_escapes) * cost_per_escape

print(f"육안 검사 연간 유출 건수: {human_escapes:,.0f}건")
print(f"AI 검사 연간 유출 건수  : {ai_escapes:,.0f}건")
print(f"연간 절감액 (추정)      : {savings:,.0f}원")

육안 검사 연간 유출 건수: 3,750건
AI 검사 연간 유출 건수  : 750건
연간 절감액 (추정)      : 150,000,000원


## 5. 데이터 요구사항 정의

### 5-1. 수집 요구사항 체크리스트

- [x] **영상 데이터**: AOI 카메라 RGB 이미지
- [x] **라벨 데이터**: YOLO bbox 어노테이션 (6-class)
- [x] **분할**: train / val / test
- [ ] **메타데이터**: 조명 조건, 해상도, 회전 (파일명에서 추출 가능)
- [ ] **센서 데이터**: 조도, 온도, 컨베이어 속도 (03번 노트북에서 시뮬레이션)

### 5-2. 📝 실습: 문제 정의서 작성

아래 빈칸을 채워 **팀별 문제 정의서**를 완성하세요.

```
[문제 정의서]
1. 현장 문제 한 줄 요약: ___________________________________
2. ML 과제 유형: ___________________________________
3. 입력 데이터: ___________________________________
4. 출력 형식: ___________________________________
5. 가장 중요한 성공 지표: ___________________________________
6. 배포 환경 제약: ___________________________________
```

---

## 다음 단계
→ **02_데이터셋_구축_EDA.ipynb**: 압축 해제된 PCB Defect 데이터셋 구조 분석, 클래스 분포, 시각화
→ **03_센서영상_문제정의_실습.ipynb**: 영상+센서 멀티모달 문제 정의 및 어노테이션 품질 검증